# Building an Agent with LangGraph and the Gemini API



## 1. Environment Setup

Install the required packages. In a Colab notebook, you can install LangGraph and the Gemini API bindings with:

```bash
!pip install -qU "langgraph==0.2.45" "langchain-google-gemini==2.0.4"
```

After installing, you need to set your Google API key in the environment. This can be done via Kaggle/Colab secrets or by storing it in a variable:

```python
import os
GOOGLE_API_KEY = "your-secret-api-key"
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
```

If you are using Kaggle, you can retrieve the key via the `UserSecretsClient`:

```python
from kaggle_secrets import UserSecretsClient

GOOGLE_API_KEY = UserSecretsClient().get_secret("GOOGLE_API_KEY")
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
```

Make sure the key is available before executing the rest of the notebook.


## 2. Define the Order State and System Instructions

LangGraph maintains an application **state** that is passed between nodes. We define a `TypedDict` called `OrderState` to hold conversation history (`messages`), the current `order`, and a `finished` flag to indicate when the conversation ends. We also create a system instruction for the Gemini model describing how the bot should behave.


In [ ]:

from typing import Annotated, TypedDict
from langgraph.graph.message import add_messages

class OrderState(TypedDict):
    '''State representing the customer's order conversation.'''
    # Messages exchanged so far
    messages: Annotated[list, add_messages]
    # Current order items
    order: list[str]
    # Flag to indicate if the conversation is finished
    finished: bool

# System instruction (playbook) guiding the bot's behaviour
BARISTABOT_SYSINT = (
    "system",  # identifies this as a system-level instruction
    "You are a BaristaBot, an interactive cafe ordering system. A human will talk to you.
"
    "Available products you have and you will answer any questions about menu items.
"
    "The customer will place an order for one or more items from the menu.
"
    "Add items to the customer's order with add_to_order, and reset the order with clear_order.
"
    "Always confirm the order before calling place_order.
"
    "Once the customer has finished ordering, call place_order.
"
    "If the customer wishes to quit, say goodbye and end the conversation."
)

# Initial welcome message
WELCOME_MSG = "Welcome to the BaristaBot cafe. Type `q` to quit. How may I serve you today?"


## 3. Define a Single Turn Chatbot Node

We define a chatbot function that takes the current `OrderState` and returns a new state with the latest model message. This function wraps the Gemini API call via LangChain's `ChatGoogleGenerativeAI`. You may need to choose an appropriate Gemini model (e.g. `gemini-1.5-flash-latest` or `gemini-pro`) depending on your quota and tool‑calling needs.


In [ ]:

from langchain_google_genai import ChatGoogleGenerativeAI

# Choose a Gemini model (flash models are fast, pro models support tool-calling)
llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash-latest")

# Chatbot node that invokes the model's chat interface
def chatbot(state: OrderState) -> OrderState:
    '''A simple chatbot node that forwards the conversation to the Gemini model.'''
    message_history = BARISTABOT_SYSINT + state["messages"]
    # Call the Gemini API using the conversation history
    response = llm.invoke(message_history)
    return {"messages": [response]}

# Build the graph and add the chatbot node as the entry point
from langgraph.graph import StateGraph, START, END

graph_builder = StateGraph(OrderState)
graph_builder.add_node("chatbot", chatbot)

# Start the graph at the chatbot node
graph_builder.add_edge(START, "chatbot")
# End the graph after chatbot (for single-turn demo)
graph_builder.add_edge("chatbot", END)

# Compile the graph
chat_graph = graph_builder.compile()

# Invoke the graph with an initial state (this call will contact the API)
initial_state: OrderState = {"messages": [], "order": [], "finished": False}
# Example call (disabled in offline environment)
# result_state = chat_graph.invoke(initial_state)
# print(result_state)


## 4. Add a Human Node and Loop the Conversation

To make the conversation interactive, we introduce a `human_node` that prints the last model message to the user and collects the user's response via `input()`. We modify the chatbot node to include a welcome message when no prior messages exist. Then we construct a graph with two nodes (`chatbot_with_welcome_msg` and `human_node`) and add edges to loop between them.


In [ ]:

from langchain_core.messages import AIMessage

# Human node: display the last model message, then request user input
def human_node(state: OrderState) -> OrderState:
    '''Display the last model message and receive user input.'''
    last_msg = state["messages"][-1]
    print("Model:", last_msg.content)
    user_input = input("User: ")
    # If the user wants to quit, set finished flag
    if user_input.lower() in ("q", "quit", "exit", "goodbye"):
        state["finished"] = True
    return {"messages": [("user", user_input)]}

# Chatbot node with a welcome message
def chatbot_with_welcome_msg(state: OrderState) -> OrderState:
    '''Wrapper around the Gemini chat interface with a welcome message on first run.'''
    if state["messages"]:
        new_output = llm.invoke([BARISTABOT_SYSINT] + state["messages"])
    else:
        new_output = AIMessage(content=WELCOME_MSG)
    return {"messages": [new_output]}

# Build a two-node graph
chat_builder = StateGraph(OrderState)
chat_builder.add_node("chatbot", chatbot_with_welcome_msg)
chat_builder.add_node("human", human_node)
chat_builder.add_edge(START, "chatbot")
chat_builder.add_edge("chatbot", "human")
chat_builder.add_edge("human", "chatbot")

# Compile the graph for looping chat
chat_with_human_graph = chat_builder.compile()

# Example invocation (requires user input)
# state = {"messages": [], "order": [], "finished": False}
# state = chat_with_human_graph.invoke(state)


## 5. Add Conditional Edges for Exit

To prevent an infinite loop, we add a conditional edge that checks the `finished` flag on the state. If `finished` is `True`, the graph transitions to `END`; otherwise, it returns to the chatbot node. LangGraph conditional edges take a function that returns the name of the next node based on the state.


In [ ]:

from typing import Literal

# Conditional routing function
def maybe_exit_human_node(state: OrderState) -> Literal["chatbot", "__end__"]:
    '''Route to chatbot or exit based on the finished flag.'''
    if state.get("finished", False):
        return END
    else:
        return "chatbot"

# Build graph with conditional edge
cond_builder = StateGraph(OrderState)
cond_builder.add_node("chatbot", chatbot_with_welcome_msg)
cond_builder.add_node("human", human_node)
cond_builder.add_edge(START, "chatbot")
cond_builder.add_edge("chatbot", "human")
# Conditional edge from human node
cond_builder.add_conditional_edges(
    "human",
    maybe_exit_human_node,
    {"chatbot": "chatbot", END: END}
)

chat_with_conditional_graph = cond_builder.compile()

# Example invocation (requires interactive input)
# state = {"messages": [], "order": [], "finished": False}
# state = chat_with_conditional_graph.invoke(state)


## 6. Add a Live Menu with Custom Tools

BaristaBot needs a live menu that can be updated dynamically. LangChain allows you to wrap Python functions as tools using the `@tool` decorator. We define two tools:

- `get_menu`: returns a list of available drinks and modifiers (stateless).
- `add_to_order`: updates the order state with the selected item (stateful).

These tools are then exposed to the Gemini model for tool‑calling. The chatbot node can call these tools automatically when it identifies a menu request or an addition to the order.


In [ ]:

from langchain_core.tools import tool

# Stateless tool to provide the current menu
@tool
def get_menu() -> str:
    return (
        "Menu: cappuccino, latte, americano, green tea, chai.
"
        "Modifiers: almond milk, oat milk, vanilla syrup, caramel syrup."
    )

# Stateful tool to add items to the order
@tool
def add_to_order(item: str, state: OrderState) -> OrderState:
    state["order"].append(item)
    return state

# To enable tool-calling, use the 'pro' Gemini model and register tools:
# llm_with_tools = ChatGoogleGenerativeAI(
#     model="gemini-1.5-pro", tools=[get_menu, add_to_order]
# )

# Then update your chatbot node to call tools based on the model output.


## 7. Observations and Next Steps

- **API Key Setup**: Ensure your Google API key is stored securely and loaded into the environment before running any model code.
- **Interactive Loop**: The human node uses `input()` for a real conversation. When running in Colab or Kaggle, you can test the loop by typing responses when prompted.
- **Graph Visualization**: LangGraph can visualize graphs via Mermaid diagrams (e.g., `chat_with_human_graph.get_graph().draw_mermaid_png()`), which helps verify node connections.
- **Live Menu**: Tools allow BaristaBot to fetch an up‑to‑date menu or modify the order. Gemini’s tool‑calling support requires the `pro` model.
- **Offline Limitations**: In this offline notebook, model invocations and tool registrations are illustrative. Execute this notebook in an online environment for full functionality.

Feel free to extend this notebook by implementing the remaining steps (visualizing the graph, creating additional state transitions, handling order confirmation, etc.) as outlined in the daily challenge instructions.
